# From Factors to Actors: a Rock-Paper-Scissors example

Statistical analysis works with **factors** - variables, and the associations between them. Agent-based modeling works with **actors** - the individuals whose decisions, repeated many times, generate the variables we later put in a table.

This notebook walks through both sides of that distinction, using a small Rock-Paper-Scissors tournament:

1. We start where a data scientist usually starts: a table, and a regression.
2. We ask what the regression can and cannot tell us about how that table came to be.
3. We build the actors - first as simple dictionaries, then as objects with a proper agent architecture, following Axtell (2000).

# Part 0 - A table, and a regression

This is the kind of data you are usually handed: one row per player, with two demographic covariates (`age`, `sex`) and a `final_points` column - the outcome of a completed Rock-Paper-Scissors tournament. Nothing here says *how* anyone actually played; we only see the result.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression

observed = pd.read_csv('rps_players.csv')
observed

,player,age,sex,final_points
0,Ava,22,F,37
1,Ben,45,M,75
2,Cleo,29,F,31
3,Dan,51,M,31
4,Eva,34,F,27
5,Finn,19,M,56
6,Gia,60,F,48
7,Hugo,38,M,69


## Regressing `final_points` on `age` and `sex`

We turn `sex` into a 0/1 column (`sex_M`, with `F` as the reference level) and fit a linear regression.

In [2]:
X = pd.get_dummies(observed[['sex']], drop_first=True, dtype=int)
X['age'] = observed['age']
y = observed['final_points']

model = LinearRegression().fit(X, y)

pd.DataFrame({
    'term': ['Intercept'] + list(X.columns),
    'coefficient': [model.intercept_] + list(model.coef_)
})

,term,coefficient
0,Intercept,32.301924
1,sex_M,21.809761
2,age,0.095119


In [3]:
print(f'R-squared: {model.score(X, y):.3f}')

R-squared: 0.415


This regression is a perfectly legitimate piece of analysis. It describes how `final_points` co-varies with `age` and `sex` in this particular table. That is all it does - and all it *can* do.

# Part 1 - What the table does not show

`final_points` is not a fact about a player. It is a **generated result**: the output of many rounds of a game, played against different opponents, decided move by move.

None of that process survives into the table. The dataframe keeps the *outcome* of the dynamics and throws away the dynamics themselves - who played whom, in what order, what each of them chose, round after round. A regression on `final_points` can only ever describe the leftover residue of that process; it has no way to reconstruct, confirm, or rule out any particular story about how the process actually worked.

To see the dynamics, we have to stop summarizing players as rows in a table and start representing them as **actors** who make a decision every time they play. That is what the rest of this notebook does.

# Part 2 - Proto-agents, as dictionaries

The simplest way to represent an actor in Python is a dictionary: a bundle of attributes that can change over time.

We build two agents directly - not by reading a row out of `observed`. They happen to carry `age` and `sex`, just so we can look at them side by side with Part 0, but watch closely: **the decision below never looks at `age` or `sex` at all.**

In [4]:
strategies = ['Rock', 'Paper', 'Scissors']

payoff = {
    ('Rock', 'Paper'): (0, 1),
    ('Paper', 'Rock'): (1, 0),
    ('Rock', 'Scissors'): (1, 0),
    ('Scissors', 'Rock'): (0, 1),
    ('Paper', 'Scissors'): (0, 1),
    ('Scissors', 'Paper'): (1, 0),
    ('Rock', 'Rock'): (0, 0),
    ('Paper', 'Paper'): (0, 0),
    ('Scissors', 'Scissors'): (0, 0)
}

In [5]:
Players = [{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 0, 'strategy': None},
           {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 0, 'strategy': None}]

Players

[{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 0, 'strategy': None},
 {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 0, 'strategy': None}]

## Decision-making process

The decision rule is as plain as it can be: pick a move at random. No reference to `age`, no reference to `sex` - the dictionary carries those fields, the decision function ignores them completely.

In [6]:
from random import Random

demo_rng = Random(7)   # a dedicated random stream, just for this demo

# simplest possible decision: choose randomly, independent of any attribute
demo_rng.choice(strategies)

'Paper'

**Each agent decides a strategy:**

In [7]:
Players[0]['strategy'] = demo_rng.choice(strategies)

In [8]:
Players[1]['strategy'] = demo_rng.choice(strategies)

**Decisions made:**

In [9]:
Players[0]['strategy'], Players[1]['strategy']

('Rock', 'Paper')

**Social result of the individual decisions:**

In [10]:
result = payoff[Players[0]['strategy'], Players[1]['strategy']]
result

(0, 1)

**Each agent's score updates:**

In [11]:
Players[0]['score'] += result[0]

In [12]:
Players[1]['score'] += result[1]

In [13]:
Players

[{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 0, 'strategy': 'Rock'},
 {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 1, 'strategy': 'Paper'}]

## Generating a final score, and comparing

One game is not a tournament. We let these same two agents play a series of rounds against each other, always with the same rule: `choice(strategies)`, nothing else. Then we compare their **generated** final scores - to each other, not to `age` or `sex`, because age and sex were never part of the mechanism.

In [14]:
# reset scores, keep identities
Players = [{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 0, 'strategy': None},
           {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 0, 'strategy': None}]

for aRound in range(1, 21):
    Players[0]['strategy'] = demo_rng.choice(strategies)
    Players[1]['strategy'] = demo_rng.choice(strategies)
    result = payoff[Players[0]['strategy'], Players[1]['strategy']]
    Players[0]['score'] += result[0]
    Players[1]['score'] += result[1]

Players

[{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 8, 'strategy': 'Scissors'},
 {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 7, 'strategy': 'Rock'}]

In [15]:
pd.DataFrame(Players)[['name', 'age', 'sex', 'score']]

,name,age,sex,score
0,Ava,22,F,8
1,Ben,45,M,7


Ava and Ben ended up with different scores after 20 rounds. Nothing in the mechanism that produced that difference had anything to do with being 22 versus 45, or `F` versus `M` - it was `choice(strategies)`, called independently each time. `age` and `sex` are just labels riding along on the dictionary; they play no causal role. Keep that firmly in mind for Part 3.

# Part 3 - Scaling up, and comparing against Part 0

We now build a proto-agent for each of the eight players in `observed`, carrying their real `name`, `age` and `sex` forward as labels - and run a full round-robin tournament with the exact same rule as Part 2: `choice(strategies)`, independent of every attribute.

In [16]:
import itertools

society = [{'name': row.player, 'age': int(row.age), 'sex': row.sex, 'score': 0, 'strategy': None}
           for row in observed.itertuples(index=False)]
society

[{'name': 'Ava', 'age': 22, 'sex': 'F', 'score': 0, 'strategy': None},
 {'name': 'Ben', 'age': 45, 'sex': 'M', 'score': 0, 'strategy': None},
 {'name': 'Cleo', 'age': 29, 'sex': 'F', 'score': 0, 'strategy': None},
 {'name': 'Dan', 'age': 51, 'sex': 'M', 'score': 0, 'strategy': None},
 {'name': 'Eva', 'age': 34, 'sex': 'F', 'score': 0, 'strategy': None},
 {'name': 'Finn', 'age': 19, 'sex': 'M', 'score': 0, 'strategy': None},
 {'name': 'Gia', 'age': 60, 'sex': 'F', 'score': 0, 'strategy': None},
 {'name': 'Hugo', 'age': 38, 'sex': 'M', 'score': 0, 'strategy': None}]

In [17]:
tournament_rng = Random(42)   # a separate stream from the Part 2 demo

for aRound in range(1, 21):
    for player1, player2 in itertools.combinations(society, 2):
        player1['strategy'] = tournament_rng.choice(strategies)
        player2['strategy'] = tournament_rng.choice(strategies)
        result = payoff[player1['strategy'], player2['strategy']]
        player1['score'] += result[0]
        player2['score'] += result[1]

generated = pd.DataFrame(society)[['name', 'age', 'sex', 'score']]
generated

,name,age,sex,score
0,Ava,22,F,53
1,Ben,45,M,39
2,Cleo,29,F,50
3,Dan,51,M,45
4,Eva,34,F,42
5,Finn,19,M,49
6,Gia,60,F,42
7,Hugo,38,M,52


## Does a regression on the *generated* scores find anything?

We run the identical regression from Part 0 - `score` on `age` and `sex` - but now on data we generated ourselves, knowing for a fact that `age` and `sex` had zero influence on the mechanism that produced it.

In [18]:
Xg = pd.get_dummies(generated[['sex']], drop_first=True, dtype=int)
Xg['age'] = generated['age']
yg = generated['score']

model_g = LinearRegression().fit(Xg, yg)

pd.DataFrame({
    'term': ['Intercept'] + list(Xg.columns),
    'coefficient': [model_g.intercept_] + list(model_g.coef_)
})

,term,coefficient
0,Intercept,55.376648
1,sex_M,-0.024047
2,age,-0.237976


In [19]:
print(f'R-squared: {model_g.score(Xg, yg):.3f}')

R-squared: 0.421


## The point

Whatever coefficients come out of the cell above are pure artifacts of randomness with a small sample - we *know* that, because we built the mechanism and `age`/`sex` never entered it. Yet the regression procedure does not know that; it will report *some* coefficient regardless, and nothing about the regression output by itself would tell you it is meaningless here.

That is exactly the limitation from Part 0. The regression on `observed` (the original table) and the regression on `generated` (our table, with a mechanism we fully control and know to be independent of age/sex) are computed by the exact same procedure and can look similarly "informative." The difference between a real effect and a coincidental one is not visible in the regression output - it depends entirely on the generating mechanism, which the table never records. This is the "factors versus actors" point from Macy and Willer (2002): a table of factors can be produced by more than one story about the actors behind it, and regression alone cannot adjudicate between them.

# Part 4 - A proper agent architecture (Axtell, 2000)

Dictionaries were enough to make the point in Parts 2-3, but Axtell (2000) describes agents more formally, as **objects**: a bundle of

- **public state** - attributes other agents or the population can see (`name`, `age`, `sex`, `score`);
- **private state** - information the agent keeps to itself (its last move);
- **behavior** - methods the agent executes on its own (`choose_move`).

We have not used Python classes yet in this course, so we build one from nothing, one piece at a time. If you already know classes, feel free to skim - the RPS logic itself does not change from Part 2/3.

## 4.1 - A class is a blueprint, an object is one instance of it

A `dict` like `{'name': 'Ava', 'score': 0}` is a one-off bundle of data. A **class** is a *template* for building many such bundles that all share the same shape and the same behavior. Each bundle built from the template is an **object** (also called an *instance*).

The emptiest possible class does nothing yet - it just lets us create objects:

In [20]:
class Player:
    pass

ava = Player()
ben = Player()

ava, type(ava)

(<__main__.Player at 0x7f4f64cf4a50>, __main__.Player)

`ava` and `ben` are two distinct objects, both built from the `Player` blueprint - but right now that blueprint gives them no attributes at all. We have to add those ourselves.

## 4.2 - `__init__` and `self`: giving every object its own attributes

`__init__` is a special method Python calls automatically every time we build an object from the class (`Player(...)`). Its job is to attach attributes to *that particular object*.

`self` is how the method refers to "this specific object" - it is always the first parameter of every method, and Python fills it in for you; you never pass it explicitly.

In [21]:
class Player:
    def __init__(self, name):
        self.name = name   # self.name = "this object's own name"

ava = Player('Ava')
ben = Player('Ben')

ava.name, ben.name

('Ava', 'Ben')

Each object keeps its *own* copy of `name` - changing `ava.name` would never touch `ben.name`. That is the whole benefit of `self`: the same `__init__` code builds independent objects.

## 4.3 - Adding the rest of the agent's public and private state

We now add every attribute the dictionaries had (`age`, `sex`, `score`), plus one attribute the dictionaries did not have: `_last_move`, prefixed with an underscore. Python does not truly enforce privacy, but the underscore is the standard convention for "this is the object's own bookkeeping - other code should not need to touch it directly.

In [22]:
class Player:
    def __init__(self, name, age, sex):
        # public state
        self.name = name
        self.age = age
        self.sex = sex
        self.score = 0
        # private state - by convention, other code leaves this alone
        self._last_move = None

    def __repr__(self):
        # controls how the object prints - convenient, not required
        return f'Player({self.name}, age={self.age}, sex={self.sex}, score={self.score})'

ava = Player('Ava', 22, 'F')
ben = Player('Ben', 45, 'M')

ava, ben

(Player(Ava, age=22, sex=F, score=0), Player(Ben, age=45, sex=M, score=0))

## 4.4 - A method: behavior that belongs to the object

A **method** is a function defined inside a class. Like `__init__`, it takes `self` first, so it can read and change that specific object's own attributes. We add `choose_move` - exactly the same rule as every decision so far in this notebook: `rng.choice(strategies)`, independent of `age` or `sex`.

In [23]:
class Player:
    def __init__(self, name, age, sex):
        self.name = name
        self.age = age
        self.sex = sex
        self.score = 0
        self._last_move = None

    def choose_move(self, rng):
        """Behavior: plain random choice - the same rule as Parts 2 and 3."""
        self._last_move = rng.choice(strategies)
        return self._last_move

    def __repr__(self):
        return f'Player({self.name}, age={self.age}, sex={self.sex}, score={self.score})'

ava = Player('Ava', 22, 'F')
ben = Player('Ben', 45, 'M')

**Calling the method** looks like `ava.choose_move(rng)`: Python passes `ava` in automatically as `self`, so we only supply `rng`.

In [24]:
oop_demo_rng = Random(7)   # its own stream, separate from every rng used earlier

ava.choose_move(oop_demo_rng)

'Paper'

In [25]:
ben.choose_move(oop_demo_rng)

'Rock'

## 4.5 - Playing one game between two `Player` objects

This is the object version of the single game we hand-ran in Part 2. Notice the payoff lookup and the score update are still ordinary code outside the class - only the *decision* was made a method.

In [26]:
result = payoff[ava._last_move, ben._last_move]
result

(1, 0)

In [27]:
ava.score += result[0]

In [28]:
ben.score += result[1]

In [29]:
ava, ben

(Player(Ava, age=22, sex=F, score=1), Player(Ben, age=45, sex=M, score=0))

## 4.6 - Giving the object a `receive_payoff` method too

Updating `.score` from outside the class works, but it means every piece of code that plays a game has to know that scores are stored as `.score` and added with `+=`. We can hide that detail inside the object as well, with one more method - this is **encapsulation**: the object manages its own state, and other code just asks for a change instead of reaching in and doing it by hand.

In [30]:
class Player:
    def __init__(self, name, age, sex):
        self.name = name
        self.age = age
        self.sex = sex
        self.score = 0
        self._last_move = None

    def choose_move(self, rng):
        self._last_move = rng.choice(strategies)
        return self._last_move

    def receive_payoff(self, points):
        self.score += points

    def __repr__(self):
        return f'Player({self.name}, age={self.age}, sex={self.sex}, score={self.score})'

## 4.7 - A `Population` class: composition, not inheritance

A `Population` is a different kind of object - one whose only job is to **hold a list of `Player` objects and control how they interact**. This is called *composition*: a `Population` is built out of `Player`s, rather than being a special kind of `Player`.

We build it in two steps, on purpose. First, the naive version - one fixed pass through `itertools.combinations`, exactly like the loop in Part 3:

In [31]:
class Population:
    def __init__(self, players):
        self.players = players

    def run_round(self, payoff, rng):
        for player1, player2 in itertools.combinations(self.players, 2):
            move1 = player1.choose_move(rng)
            move2 = player2.choose_move(rng)
            points1, points2 = payoff[move1, move2]
            player1.receive_payoff(points1)
            player2.receive_payoff(points2)

## 4.8 - Why that naive version is not good enough

`itertools.combinations(self.players, 2)` always visits the *same* pairs in the *same* order, round after round. That fixed order is not part of the social process we are modeling - it is a side effect of how we happened to write the loop. Axtell (2000) warns about exactly this:

> "the order of agent activation must be systematically randomized from period to period in order to avoid the production of artifacts."

If we never shuffle, whichever player happens to always go first in the pairing order could end up with a small structural edge, purely from the implementation - not from anything about the game. We fix this once, inside `run_round`, instead of trusting every future loop to remember it:

In [32]:
class Population:
    """Holds the agents and controls how they interact each period."""

    def __init__(self, players):
        self.players = players

    def run_round(self, payoff, rng):
        # Axtell (2000): randomize activation order every period,
        # so the pairing schedule itself never becomes part of the story.
        order = self.players[:]
        rng.shuffle(order)
        for player1, player2 in itertools.combinations(order, 2):
            move1 = player1.choose_move(rng)
            move2 = player2.choose_move(rng)
            points1, points2 = payoff[move1, move2]
            player1.receive_payoff(points1)
            player2.receive_payoff(points2)

    def run_tournament(self, rounds, payoff, rng):
        for _ in range(rounds):
            self.run_round(payoff, rng)

    def as_dataframe(self):
        return pd.DataFrame([
            {'name': p.name, 'age': p.age, 'sex': p.sex, 'score': p.score}
            for p in self.players
        ])

## 4.9 - Running the same tournament through the object-based agents

In [33]:
players_oop = [Player(row.player, int(row.age), row.sex) for row in observed.itertuples(index=False)]
population = Population(players_oop)

oop_rng = Random(99)   # its own stream, separate from Part 2, Part 3, and the 4.4 demo
population.run_tournament(rounds=20, payoff=payoff, rng=oop_rng)

population.as_dataframe()

,name,age,sex,score
0,Ava,22,F,44
1,Ben,45,M,45
2,Cleo,29,F,58
3,Dan,51,M,53
4,Eva,34,F,48
5,Finn,19,M,49
6,Gia,60,F,41
7,Hugo,38,M,44


## 4.10 - Inheritance: reusing a class instead of rewriting it

One more core OOP idea, briefly. `class AlwaysRock(Player)` means "an `AlwaysRock` *is a* `Player`, with everything `Player` has, except for whatever we override below." We only need to override `choose_move` - `__init__`, `receive_payoff`, `__repr__` are all inherited unchanged.

In [34]:
class AlwaysRock(Player):
    def choose_move(self, rng):
        # override: ignore rng entirely, always play Rock
        self._last_move = 'Rock'
        return self._last_move

stubborn = AlwaysRock('Stubborn', age=30, sex='M')
stubborn.choose_move(oop_demo_rng), stubborn.choose_move(oop_demo_rng), stubborn.choose_move(oop_demo_rng)

('Rock', 'Rock', 'Rock')

`stubborn` can be handed to `Population` exactly like any other `Player` - `run_round` calls `player.choose_move(rng)` without caring whether that object is a plain `Player` or an `AlwaysRock`. This is **polymorphism**: the population code did not change at all, yet the agents' behavior can now differ by class.

In [35]:
mixed_society = [Player('Ava', 22, 'F'), Player('Ben', 45, 'M'), AlwaysRock('Stubborn', 30, 'M')]
mixed_population = Population(mixed_society)

mixed_rng = Random(5)
mixed_population.run_tournament(rounds=20, payoff=payoff, rng=mixed_rng)

mixed_population.as_dataframe()

,name,age,sex,score
0,Ava,22,F,13
1,Ben,45,M,16
2,Stubborn,30,M,10


### What this section taught, and what it did not

We used the RPS agent to introduce five core OOP ideas: **class vs. object** (4.1), **`__init__`/`self`** (4.2-4.3), **methods** as an object's own behavior (4.4-4.6), **composition** (`Population` built out of `Player`s, 4.7-4.9), and **inheritance/polymorphism** (`AlwaysRock`, 4.10).

None of this made the *decisions* any smarter than Part 2's `choice(strategies)` - `Player.choose_move` is still plain randomness. That is deliberate: the architecture (where behavior and state live, who is responsible for the schedule) is a separate question from what the behavior actually is. The next step in the course is to change what happens inside `choose_move` - now that there is a well-defined, single place to change it.